# Modélisation – H3a : l'attractivité (équipements, délinquance)
**Contributeur :** Théophyle · **Date :** 05/11/2026

Notebook construit à partir du template commun de l'équipe. Il passe par `commun.py` pour tout ce qui doit être identique : la table, le masque des lignes, les fenêtres, la naïve, `evaluer` et `tester`. Le gradient boosting utilise le gabarit de l'équipe (`gabarit_boosting.py`, P3).

**Question :** ajoutée au prix N-1, l'attractivité d'un territoire (densité, équipements, délinquance) améliore-t-elle la prévision du prix médian au m² ?

**Attendu (feuille de route) :** un petit gain pour les appartements, aucun pour les maisons. L'attractivité se voit surtout en ville (commerces, transports, vie nocturne), là où les appartements sont nombreux.

## 0. Paramètres de l'hypothèse
Les variables sont rangées en trois groupes :
- **densité** : habitants par km². Elle est dans tous les modèles du bloc : sans elle, les équipements par habitant mélangeraient l'offre et le caractère rural (un stade pour 300 habitants donne un taux élevé, sans que le territoire soit attractif) ;
- **équipements** : 6 domaines de la base permanente des équipements, pour 10 000 habitants ;
- **délinquance** : 7 indicateurs, en taux lissés sur 3 ans (les destructions ont été retirées de la table, redondantes avec les vols sans violence).

`VARIABLES_LOG` et `VARIABLES_LOG1P` : les variables asymétriques passées en log (règle de l'audit complémentaire : asymétrie > 1), pour les seuls modèles qui en ont besoin (régressions). Toutes les variables du bloc sont concernées. `prix_n1` passe aussi en log, avec la cible, comme dans H3b.

In [1]:
HYPOTHESE = "H3a"

DENSITE = ["densite_population"]
EQUIPEMENTS = ["taux_equip_commerces", "taux_equip_sante", "taux_equip_services",
               "taux_equip_sports", "taux_equip_tourisme", "taux_equip_transports"]
DELINQUANCE = ["taux3_vols_vehicules", "taux3_vols_violents", "taux3_vols_sans_violence",
               "taux3_cambriolages", "taux3_stupefiants",
               "taux3_violences_intrafamiliales", "taux3_violences_sexuelles"]
VARIABLES_BLOC = DENSITE + EQUIPEMENTS + DELINQUANCE     # tous les groupes réunis (constat rapide, section 2)

# Variables asymétriques de la table (audit complémentaire, version 2)
LOG = ["nb_ventes_n1", "densite_population", "niveau_vie_median",
       "taux3_vols_sans_violence", "taux3_vols_vehicules",
       "taux3_violences_sexuelles", "taux3_cambriolages", "taux3_violences_intrafamiliales",
       "taux_equip_commerces", "taux_equip_services", "taux_equip_sante", "taux_equip_sports",
       "part_sans_voiture", "part_moins40m2", "part_rsec", "part_apparts",
       "part_familles_monoparentales_pct_fam"]
LOG1P = ["taux3_vols_violents", "taux3_stupefiants", "taux_equip_tourisme", "taux_equip_transports"]

# Variables de l'hypothèse seulement ; prix_n1 passe en log avec la cible
VARIABLES_LOG = [v for v in LOG if v in VARIABLES_BLOC] + ["prix_n1"]
VARIABLES_LOG1P = [v for v in LOG1P if v in VARIABLES_BLOC]
print("log :", VARIABLES_LOG)
print("log(1 + x) :", VARIABLES_LOG1P)

log : ['densite_population', 'taux3_vols_sans_violence', 'taux3_vols_vehicules', 'taux3_violences_sexuelles', 'taux3_cambriolages', 'taux3_violences_intrafamiliales', 'taux_equip_commerces', 'taux_equip_services', 'taux_equip_sante', 'taux_equip_sports', 'prix_n1']
log(1 + x) : ['taux3_vols_violents', 'taux3_stupefiants', 'taux_equip_tourisme', 'taux_equip_transports']


## 1. Import et chargement
`charger_table()` retire une seule fois les lignes à case vide (masque commun) : tous les modèles et la naïve sont jugés sur les mêmes lignes. `fenetres()` donne une fenêtre par année jugée (2023 et 2024), avec l'apprentissage sur les années d'avant.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from copy import deepcopy
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import commun
from commun import CIBLE, charger_table, fenetres, plis_temporels, evaluer, tester
from gabarit_hierarchique import ModeleHierarchique

table = charger_table()
FENETRES = fenetres(table)

absentes = sorted(set(VARIABLES_BLOC + VARIABLES_LOG + VARIABLES_LOG1P) - set(table.columns))
assert not absentes, f"Variables absentes de la table : {absentes}"
print("Table :", table.shape)
for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {sorted(app['annee'].unique())} ({len(app)} lignes) | jugée {a} ({len(ev)} lignes)")

Table : (23489, 39)
Fenêtre 2023 : apprentissage [np.int64(2020), np.int64(2021), np.int64(2022)] (7987 lignes) | jugée 2023 (2641 lignes)
Fenêtre 2024 : apprentissage [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] (10628 lignes) | jugée 2024 (2599 lignes)


## 2. Constat rapide
Distribution des variables du bloc et lien avec le prix (rho de Spearman), sur les années d'apprentissage de la première fenêtre seulement : l'année jugée n'est jamais regardée avant la modélisation.

In [3]:
app_constat = FENETRES[commun.ANNEES_JUGEES[0]][0]
print(app_constat[VARIABLES_BLOC].describe().round(2).to_string())

liens = []
for type_bien, e in app_constat.groupby("type_bien"):
    for v in VARIABLES_BLOC:
        liens.append({"type de bien": type_bien, "variable": v, "rho": stats.spearmanr(e[v], e[CIBLE]).statistic})
pd.DataFrame(liens).pivot(index="variable", columns="type de bien", values="rho").round(2)

       densite_population  taux_equip_commerces  taux_equip_sante  taux_equip_services  taux_equip_sports  taux_equip_tourisme  taux_equip_transports  taux3_vols_vehicules  taux3_vols_violents  taux3_vols_sans_violence  taux3_cambriolages  taux3_stupefiants  taux3_violences_intrafamiliales  taux3_violences_sexuelles
count             7987.00               7987.00           7987.00              7987.00            7987.00              7987.00                7987.00               7987.00              7987.00                   7987.00             7987.00            7987.00                          7987.00                    7987.00
mean               527.22                 57.38             52.92               217.22              39.14                 9.82                   6.90                  4.03                 0.48                      6.61                5.25               2.73                             1.95                       1.07
std               2451.18                 40.9

type de bien,appartement,maison
variable,,
densite_population,0.41,0.65
taux3_cambriolages,0.12,0.21
taux3_stupefiants,0.10,0.16
taux3_violences_intrafamiliales,-0.20,-0.15
taux3_violences_sexuelles,-0.18,-0.26
taux3_vols_sans_violence,0.32,0.29
taux3_vols_vehicules,0.27,0.42
taux3_vols_violents,0.25,0.40
taux_equip_commerces,0.19,0.12


**Comment lire le tableau :** le rho de Spearman entre chaque variable et le prix, sur les années d'apprentissage de la fenêtre 2023 (2020-2022). Il mesure le lien avec le **niveau** du prix : un territoire dense ou bien équipé est-il plus cher ?

**Constat :**
- **La densité est la variable la plus liée au niveau du prix** (rho = 0,65 pour les maisons, 0,41 pour les appartements), avec les équipements de transport (0,51 et 0,48).
- **Les équipements de sport sont liés négativement** (−0,62 et −0,46) : rapportés à la population, ils signalent les territoires ruraux, où les prix sont bas.
- **La délinquance est faiblement liée au prix.** Les vols (véhicules, violents, sans violence) vont avec des prix plus élevés (0,25 à 0,42), car ils sont fréquents en ville ; les violences intrafamiliales et sexuelles vont avec des prix plus bas (−0,15 à −0,26).
- Les liens sont un peu plus forts pour les maisons que pour les appartements.

### 2.2 Redondances dans le bloc et lien avec l'écart au prix N-1
- **Redondances** : couples de variables du bloc dont |rho| dépasse 0,7 ; deux variables aussi liées portent la même information.
- **Écart au prix N-1** : `prix_n1` porte déjà le niveau du prix. Ce que le bloc peut apporter, c'est ce que `prix_n1` n'explique pas : l'évolution relative prix / prix N-1 − 1. On mesure son lien avec chaque variable (rho de Spearman).
- Années d'apprentissage de la première fenêtre seulement. Un même territoire revient chaque année : les rho décrivent, ils ne se testent pas.

In [4]:
couples = []
for type_bien, e in app_constat.groupby("type_bien"):
    rho = e[VARIABLES_BLOC].corr(method="spearman")
    for i, a in enumerate(VARIABLES_BLOC):
        for b in VARIABLES_BLOC[i + 1:]:
            if abs(rho.loc[a, b]) > 0.7:
                couples.append({"type de bien": type_bien, "variable 1": a, "variable 2": b, "rho": round(rho.loc[a, b], 2)})
print(pd.DataFrame(couples).to_string(index=False) if couples else "Aucun couple au-delà de 0,7")

evolution_app = app_constat[CIBLE] / app_constat["prix_n1"] - 1
liens_evolution = []
for type_bien, e in app_constat.groupby("type_bien"):
    for v in VARIABLES_BLOC:
        liens_evolution.append({"type de bien": type_bien, "variable": v,
                                "rho": stats.spearmanr(e[v], evolution_app.loc[e.index]).statistic})
pd.DataFrame(liens_evolution).pivot(index="variable", columns="type de bien", values="rho").round(2)

type de bien           variable 1          variable 2   rho
 appartement   densite_population   taux_equip_sports -0.76
 appartement taux_equip_commerces taux_equip_tourisme  0.71
      maison   densite_population   taux_equip_sports -0.74


type de bien,appartement,maison
variable,,
densite_population,-0.01,0.05
taux3_cambriolages,-0.15,-0.13
taux3_stupefiants,-0.07,-0.05
taux3_violences_intrafamiliales,0.01,0.01
taux3_violences_sexuelles,0.02,0.03
taux3_vols_sans_violence,-0.03,-0.02
taux3_vols_vehicules,-0.10,-0.06
taux3_vols_violents,-0.11,-0.06
taux_equip_commerces,0.04,0.04


**Comment lire les tableaux :**
- **Redondances** : les couples de variables du bloc liées au-delà de |rho| = 0,7. Dans une régression, deux variables très liées se partagent leur effet, et leurs coefficients deviennent difficiles à lire.
- **Lien avec l'écart au prix N-1** : le rho entre chaque variable et l'évolution du prix (prix / prix N-1 − 1). C'est la question de l'hypothèse : le prix N-1 donne déjà le niveau, le bloc ne peut aider qu'en distinguant les territoires qui montent plus que les autres. Un rho proche de 0 annonce un apport faible.

**Constat :**
- **Deux couples dépassent 0,7** : densité / équipements de sport (−0,74 pour les maisons, −0,76 pour les appartements) et, pour les appartements, commerces / tourisme (0,71). Dans une régression, ces variables se partagent leur effet : leurs coefficients ne se lisent pas séparément.
- **Aucune variable n'est liée à l'évolution du prix** : tous les rho sont compris entre −0,15 et +0,06. Les plus forts sont négatifs et viennent de la délinquance (cambriolages −0,15 et −0,13, vols violents −0,11 pour les appartements) : les territoires où elle est plus élevée ont un peu moins monté.
- **Conséquence attendue :** le prix N-1 donne déjà le niveau, et le bloc ne distingue presque pas les territoires qui montent plus que les autres. Son apport ne peut être que faible.

## 3. Fonctions du notebook
Chaque modèle est lancé en une ligne : `lancer(nom, modèle, variables, log=..., grille=...)`.
- `nom` : le nom du modèle dans les tableaux (le préfixe de l'hypothèse est ajouté automatiquement) ;
- `modèle` : le modèle à utiliser (régression linéaire, Lasso, gradient boosting, modèle hiérarchique) ;
- `variables` : les colonnes de la table données au modèle ;
- `log=True` : la cible et les variables de `VARIABLES_LOG` passent en log, celles de `VARIABLES_LOG1P` en log(1 + x) ; à réserver aux modèles qui en ont besoin (régressions, Lasso, modèle hiérarchique), jamais aux arbres ;
- `grille` : les valeurs d'un réglage à tester (par exemple `alpha` pour le Lasso).

Ce que fait `lancer`, pour chacune des deux années jugées (2023, puis 2024) et pour chaque type de bien :
1. elle prend les années d'apprentissage, avant l'année jugée ;
2. elle met les variables à l'échelle et entraîne le modèle ; avec une grille, elle choisit d'abord le meilleur réglage, chaque année d'apprentissage servant tour à tour de validation (`plis_temporels` de `commun.py`) ;
3. elle prévoit le prix de l'année jugée, remis en €/m² si le modèle a appris en log ;
4. elle compare la prévision au prix réel et à la naïve (prix N = prix N-1) avec `evaluer` et `tester` de `commun.py`.

Elle affiche ensuite l'erreur du modèle (MAE, en €/m²), celle de la naïve, le gain sur la naïve (négatif : le modèle fait moins bien) et la part des sous-bassins où le modèle gagne. Elle garde les modèles entraînés (`ajustes`, pour la section 6) et leurs prévisions (`previsions`, pour la section 5.2). Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois.

Le modèle hiérarchique (`gabarit_hierarchique.py`) passe par une enveloppe : il n'est pas mis dans le pipeline (il met lui-même ses variables à l'échelle) et n'accepte pas de grille.

In [5]:
resultats, tests, ajustes, previsions = [], [], {}, {}


def en_log(X):
    """Variables asymétriques en log ; log(1 + x) pour celles qui ont des zéros."""
    X = X.copy()
    cols = [c for c in VARIABLES_LOG if c in X.columns]
    cols1p = [c for c in VARIABLES_LOG1P if c in X.columns]
    assert (X[cols] > 0).all().all(), "valeur nulle ou négative dans une variable de VARIABLES_LOG : la passer dans LOG1P"
    X[cols] = np.log(X[cols])
    X[cols1p] = np.log1p(X[cols1p])
    return X


class Hierarchique(RegressorMixin, BaseEstimator):
    """Enveloppe scikit-learn du modèle hiérarchique : log de la cible si demandé, prévisions en €/m²."""
    def __init__(self, modele=None, log=False):
        self.modele = modele
        self.log = log

    def fit(self, X, y):
        self.modele_ = deepcopy(self.modele).fit(X, np.log(y) if self.log else y)
        return self

    def predict(self, X):
        p = np.asarray(self.modele_.predict(X))
        return np.exp(p) if self.log else p


def lancer(nom, modele, variables, log=False, grille=None):
    """Ajuste un modèle par fenêtre et par type de bien, puis evaluer et tester ; affiche le résultat."""
    nom = f"{HYPOTHESE} – {nom}"
    # Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois
    resultats[:] = [r for r in resultats if r["modèle"] != nom]
    tests[:] = [t for t in tests if t["modèle"] != nom]
    for annee, (app, ev) in FENETRES.items():
        prevision = pd.Series(np.nan, index=ev.index)
        for type_bien, e in app.groupby("type_bien"):
            idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
            X_app, X_ev = e[variables], ev.loc[idx, variables]
            if isinstance(modele, ModeleHierarchique):
                assert not grille, "pas de grille pour le modèle hiérarchique"
                if log:
                    X_app, X_ev = en_log(X_app), en_log(X_ev)
                m = Hierarchique(modele, log).fit(X_app, e[CIBLE])
                prevision.loc[idx] = m.predict(X_ev)
                ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
                continue
            m = Pipeline([("echelle", StandardScaler()), ("modele", clone(modele))])
            prefixe = "modele__"
            if log:
                X_app, X_ev = en_log(X_app), en_log(X_ev)
                m = TransformedTargetRegressor(regressor=m, func=np.log, inverse_func=np.exp)
                prefixe = "regressor__modele__"
            if grille:
                # plis recalculés sur e (positions dans e, même ordre que X_app)
                m = GridSearchCV(m, {prefixe + k: v for k, v in grille.items()},
                                 cv=plis_temporels(e), scoring="neg_mean_absolute_error")
            m.fit(X_app, e[CIBLE])
            prevision.loc[idx] = m.predict(X_ev)
            ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
        assert prevision.notna().all(), f"{nom} : prévisions manquantes"
        previsions[(nom, annee)] = prevision          # gardée pour comparer les modèles entre eux (section 5.2)
        evaluer(nom, ev, prevision, resultats)
        tester(nom, ev, prevision, tests)

    # Résultat du modèle, affiché tout de suite : erreur du modèle, erreur de la naïve, gain
    res = pd.DataFrame(resultats)
    res = res[res["modèle"] == nom][["type de bien", "année jugée", "MAE (€/m²)", "MAE naïve (€/m²)",
                                     "gain sur la naïve (%)", "bassins où le modèle gagne (%)"]]
    print(nom, f"({len(variables)} variables)")
    print(res.round(1).to_string(index=False))
    return nom

### Le gradient boosting de l'équipe : `lancer_boosting`
`lancer` met toujours une mise à l'échelle avant le modèle ; elle transformerait `prix_n1`, dont le gabarit a besoin pour remettre sa prévision en €/m². `lancer_boosting` fait donc la même chose que `lancer` (deux fenêtres, deux types de bien, `evaluer`, `tester`, résultats rangés pour la section 5.2), mais avec `prevoir_boosting` de `gabarit_boosting.py` :
- réglage sur les plis temporels, avec la grille commune de l'équipe ;
- apprentissage de l'évolution du prix (log(prix / prix N-1)), prévision remise en €/m² ;
- pas de log des variables : inutile pour des arbres.

In [6]:
from gabarit_boosting import prevoir_boosting


def lancer_boosting(nom, variables):
    """Comme lancer, avec le gabarit gradient boosting de l'équipe (P3)."""
    nom = HYPOTHESE + " – " + nom
    # Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois
    resultats[:] = [r for r in resultats if r["modèle"] != nom]
    tests[:] = [t for t in tests if t["modèle"] != nom]
    for annee, (app, ev) in FENETRES.items():
        # Réglage, réapprentissage et prévision de l'année jugée, un modèle par type de bien
        prevision, modeles = prevoir_boosting(app, ev, variables)
        for type_bien, m in modeles.items():
            idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
            ajustes[(nom, annee, type_bien)] = (m, ev.loc[idx, variables], ev.loc[idx, CIBLE])
        previsions[(nom, annee)] = prevision          # gardée pour la comparaison à la référence (section 5.2)
        evaluer(nom, ev, prevision, resultats)
        tester(nom, ev, prevision, tests)

    # Résultat du modèle, affiché tout de suite
    res = pd.DataFrame(resultats)
    res = res[res["modèle"] == nom][["type de bien", "année jugée", "MAE (€/m²)", "MAE naïve (€/m²)",
                                     "gain sur la naïve (%)", "bassins où le modèle gagne (%)"]]
    print(nom, "(" + str(len(variables)) + " variables)")
    print(res.round(1).to_string(index=False))
    return nom

## 4. Modèles
La feuille de route demande, en régression linéaire puis en gradient boosting : prix N-1 + le bloc entier, prix N-1 + équipements, prix N-1 + délinquance. On ajoute **prix N-1 + densité seule**, pour savoir si les équipements et la délinquance apportent quelque chose au-delà de la densité (qui est aussi testée en variante dans H3b).

| Modèle | Variables ajoutées au prix N-1 | Question |
|---|---|---|
| référence | aucune | ce que le bloc doit battre |
| densité | densité | la densité seule aide-t-elle ? |
| équipements | densité + 6 équipements | l'offre d'équipements aide-t-elle ? |
| délinquance | densité + 7 délinquances | la délinquance aide-t-elle ? |
| bloc complet | densité + équipements + délinquance | tout le bloc |

Chaque combinaison est lancée en régression linéaire (en log), puis avec le gradient boosting de l'équipe. Le Lasso, sur le bloc complet, dit quelles variables survivent.

In [7]:
# Combinaisons de variables : la densité est dans chaque modèle du bloc
REFERENCE_VARS = ["prix_n1"]
BLOC_DENSITE = REFERENCE_VARS + DENSITE
BLOC_EQUIPEMENTS = REFERENCE_VARS + DENSITE + EQUIPEMENTS
BLOC_DELINQUANCE = REFERENCE_VARS + DENSITE + DELINQUANCE
BLOC_COMPLET = REFERENCE_VARS + VARIABLES_BLOC

for nom, b in [("densité", BLOC_DENSITE), ("équipements", BLOC_EQUIPEMENTS),
               ("délinquance", BLOC_DELINQUANCE), ("complet", BLOC_COMPLET)]:
    print(nom, ":", len(b), "variables")

densité : 2 variables
équipements : 8 variables
délinquance : 9 variables
complet : 15 variables


### 4.1 Référence : prix N-1 seul
La naïve (prix N = prix N-1) est déjà dans chaque ligne du tableau (« MAE naïve »). Le modèle « prix N-1 seul » est la référence que le bloc doit battre.

In [8]:
REFERENCE = lancer("Régression linéaire – prix N-1 seul", LinearRegression(), REFERENCE_VARS, log=True)

H3a – Régression linéaire – prix N-1 seul (1 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       185.5             166.3                  -11.5                            45.6
      maison         2023       170.0             111.7                  -52.2                            31.0
 appartement         2024       221.5             148.0                  -49.7                            27.0
      maison         2024       214.9             129.2                  -66.4                            18.2


### 4.2 Régression linéaire : + densité

In [9]:
lancer("Régression linéaire – prix N-1 + densité", LinearRegression(), BLOC_DENSITE, log=True)

H3a – Régression linéaire – prix N-1 + densité (2 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       186.8             166.3                  -12.3                            45.9
      maison         2023       170.0             111.7                  -52.2                            31.0
 appartement         2024       220.9             148.0                  -49.2                            27.0
      maison         2024       214.5             129.2                  -66.0                            18.4


'H3a – Régression linéaire – prix N-1 + densité'

### 4.3 Régression linéaire : + densité + équipements

In [10]:
lancer("Régression linéaire – prix N-1 + densité + équipements", LinearRegression(), BLOC_EQUIPEMENTS, log=True)

H3a – Régression linéaire – prix N-1 + densité + équipements (8 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       186.4             166.3                  -12.1                            46.5
      maison         2023       167.6             111.7                  -50.1                            30.8
 appartement         2024       222.2             148.0                  -50.1                            26.2
      maison         2024       209.8             129.2                  -62.4                            18.7


'H3a – Régression linéaire – prix N-1 + densité + équipements'

### 4.4 Régression linéaire : + densité + délinquance

In [11]:
lancer("Régression linéaire – prix N-1 + densité + délinquance", LinearRegression(), BLOC_DELINQUANCE, log=True)

H3a – Régression linéaire – prix N-1 + densité + délinquance (9 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       193.5             166.3                  -16.3                            43.9
      maison         2023       186.1             111.7                  -66.6                            28.3
 appartement         2024       211.7             148.0                  -43.0                            27.6
      maison         2024       199.6             129.2                  -54.5                            20.6


'H3a – Régression linéaire – prix N-1 + densité + délinquance'

### 4.5 Régression linéaire : bloc complet

In [12]:
COMPLET = lancer("Régression linéaire – bloc complet", LinearRegression(), BLOC_COMPLET, log=True)

H3a – Régression linéaire – bloc complet (15 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       194.8             166.3                  -17.1                            43.4
      maison         2023       184.4             111.7                  -65.1                            28.1
 appartement         2024       214.7             148.0                  -45.1                            26.9
      maison         2024       198.3             129.2                  -53.5                            20.4


### 4.6 Lasso : quelles variables survivent
Le Lasso ramène à zéro les coefficients des variables qui n'apportent rien. `alpha` est réglé sur les plis temporels. Les valeurs testées sont petites : la cible est en log et les variables sont mises à l'échelle, les coefficients sont donc petits eux aussi.

In [13]:
LASSO = lancer("Lasso – bloc complet", Lasso(max_iter=10_000), BLOC_COMPLET, log=True,
               grille={"alpha": [0.0001, 0.001, 0.01, 0.1]})

H3a – Lasso – bloc complet (15 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       174.0             166.3                   -4.6                            47.7
      maison         2023       183.7             111.7                  -64.4                            28.1
 appartement         2024       200.3             148.0                  -35.3                            27.7
      maison         2024       198.3             129.2                  -53.5                            18.0


### 4.7 Gradient boosting (gabarit de l'équipe)
Mêmes combinaisons qu'en régression. La référence du boosting est **le boosting sur prix N-1 seul** : en section 5.2, chaque boosting est comparé à elle, pour ne comparer que des jeux de variables, pas deux types de modèles. Environ une minute par modèle.

In [14]:
REFERENCE_GB = lancer_boosting("Gradient boosting – prix N-1 seul", REFERENCE_VARS)
lancer_boosting("Gradient boosting – prix N-1 + densité", BLOC_DENSITE)
lancer_boosting("Gradient boosting – prix N-1 + densité + équipements", BLOC_EQUIPEMENTS)
lancer_boosting("Gradient boosting – prix N-1 + densité + délinquance", BLOC_DELINQUANCE)
COMPLET_GB = lancer_boosting("Gradient boosting – bloc complet", BLOC_COMPLET)

H3a – Gradient boosting – prix N-1 seul (1 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       182.6             166.3                   -9.8                            45.8
      maison         2023       171.5             111.7                  -53.5                            31.2
 appartement         2024       222.8             148.0                  -50.5                            26.4
      maison         2024       216.7             129.2                  -67.7                            18.5
H3a – Gradient boosting – prix N-1 + densité (2 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       183.6             166.3                  -10.4                            45.5
      maison         2023       172.5             111.7                  -54.4                            31.2

## 5. Mesures et tests
Trois tableaux, du plus simple au plus exigeant :
1. **`res` : les erreurs.** MAE du modèle et de la naïve en €/m², gain sur la naïve en %, part des sous-bassins où le modèle gagne. C'est une description : un modèle peut avoir une MAE plus basse par hasard.
2. **`tst` : les tests contre la naïve.** Ils disent si l'écart avec la naïve est plus grand que le hasard.
   - **Écart de MAE avec la naïve** : erreur du modèle moins erreur de la naïve, en €/m² (négatif : le modèle fait mieux).
   - **IC 95 %** (bootstrap par bassin de vie) : la fourchette dans laquelle se trouve cet écart, en tenant compte du hasard. Si toute la fourchette est sous 0, le modèle fait vraiment mieux ; si elle contient 0, on ne peut pas conclure ; si elle est toute au-dessus de 0, il fait vraiment moins bien.
   - **p-value de Wilcoxon** : le modèle fait-il mieux (ou moins bien) dans la plupart des sous-bassins ? Avec plusieurs centaines de sous-bassins, elle est presque toujours très petite : on la lit avec la taille d'effet.
   - **Rang-bisériale** (taille d'effet) : de −1 (le modèle fait mieux partout) à +1 (il fait moins bien partout) ; autour de 0, pas de différence nette.
3. **`decision` : la règle de décision.** Une année est réussie si l'IC est entièrement sous 0, la rang-bisériale négative et le gain d'au moins 3 %. Un modèle est retenu s'il réussit la majorité des années jugées (ici, les deux).

In [15]:
res = pd.DataFrame(resultats)
res.round(2)

,modèle,type de bien,année jugée,MAE (€/m²),MAE en % du prix,MAE naïve (€/m²),gain sur la naïve (%),bassins où le modèle gagne (%)
0,H3a – Régression linéaire – prix N-1 seul,appartement,2023,185.46,7.07,166.33,-11.51,45.64
1,H3a – Régression linéaire – prix N-1 seul,maison,2023,170.00,7.76,111.71,-52.18,30.96
2,H3a – Régression linéaire – prix N-1 seul,appartement,2024,221.51,8.50,148.00,-49.67,26.97
3,H3a – Régression linéaire – prix N-1 seul,maison,2024,214.92,9.64,129.19,-66.36,18.24
4,H3a – Régression linéaire – prix N-1 + densité,appartement,2023,186.83,7.09,166.33,-12.33,45.88
5,H3a – Régression linéaire – prix N-1 + densité,maison,2023,169.99,7.75,111.71,-52.17,30.96
6,H3a – Régression linéaire – prix N-1 + densité,appartement,2024,220.86,8.49,148.00,-49.23,26.97
7,H3a – Régression linéaire – prix N-1 + densité,maison,2024,214.50,9.62,129.19,-66.04,18.41
8,H3a – Régression linéaire – prix N-1 + densité...,appartement,2023,186.44,7.00,166.33,-12.10,46.46
9,H3a – Régression linéaire – prix N-1 + densité...,maison,2023,167.62,7.63,111.71,-50.05,30.84


In [16]:
tst = pd.DataFrame(tests)
tst.round(3)

,modèle,type de bien,année jugée,écart de MAE avec la naïve (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale,bassins de vie
0,H3a – Régression linéaire – prix N-1 seul,appartement,2023,19.140,-2.677,45.770,0.004,0.114,726
1,H3a – Régression linéaire – prix N-1 seul,maison,2023,58.291,50.608,65.476,0.000,0.471,1606
2,H3a – Régression linéaire – prix N-1 seul,appartement,2024,73.509,59.615,87.685,0.000,0.606,692
3,H3a – Régression linéaire – prix N-1 seul,maison,2024,85.732,80.270,91.580,0.000,0.778,1605
4,H3a – Régression linéaire – prix N-1 + densité,appartement,2023,20.505,-1.916,48.148,0.003,0.116,726
5,H3a – Régression linéaire – prix N-1 + densité,maison,2023,58.277,50.476,65.633,0.000,0.472,1606
6,H3a – Régression linéaire – prix N-1 + densité,appartement,2024,72.861,60.152,85.598,0.000,0.605,692
7,H3a – Régression linéaire – prix N-1 + densité,maison,2024,85.309,79.699,91.321,0.000,0.779,1605
8,H3a – Régression linéaire – prix N-1 + densité...,appartement,2023,20.117,-4.832,51.895,0.009,0.103,726
9,H3a – Régression linéaire – prix N-1 + densité...,maison,2023,55.915,48.216,63.251,0.000,0.465,1606


### 5.1 Règle de décision contre la naïve

In [17]:
SEUIL_GAIN = 3      # % de gain sur la MAE de la comparaison, fixé avant les résultats


def regle_decision(res, tst, comparaison="naïve"):
    """Années réussies et décision, par modèle et type de bien."""
    cles = ["modèle", "type de bien", "année jugée"]
    regle = res[cles + [f"gain sur la {comparaison} (%)"]].merge(
        tst[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
    regle["année réussie"] = ((regle["IC 95 % haut (€/m²)"] < 0) & (regle["rang-bisériale"] < 0)
                              & (regle[f"gain sur la {comparaison} (%)"] >= SEUIL_GAIN))
    decision = regle.groupby(["modèle", "type de bien"], sort=False)["année réussie"].agg(["sum", "count"])
    decision["retenu"] = decision["sum"] > decision["count"] / 2
    return decision.rename(columns={"sum": "années réussies", "count": "années jugées"})


regle_decision(res, tst)

années réussies  \
modèle                                             type de bien                    
H3a – Régression linéaire – prix N-1 seul          appartement                 0   
                                                   maison                      0   
H3a – Régression linéaire – prix N-1 + densité     appartement                 0   
                                                   maison                      0   
H3a – Régression linéaire – prix N-1 + densité ... appartement                 0   
                                                   maison                      0   
H3a – Régression linéaire – prix N-1 + densité ... appartement                 0   
                                                   maison                      0   
H3a – Régression linéaire – bloc complet           appartement                 0   
                                                   maison                      0   
H3a – Lasso – bloc complet                         appartement                 0   
                                                   maison                      0   
H3a – Gradient boosting – prix N-1 seul            appartement                 0   
                                                   maison                      0   
H3a – Gradient boosting – prix N-1 + densité       appartement                 0   
                                                   maison                      0   
H3a – Gradient boosting – prix N-1 + densité + ... appartement                 0   
                                                   maison                      0   
H3a – Gradient boosting – prix N-1 + densité + ... appartement                 0   
                                                   maison                      0   
H3a – Gradient boosting – bloc complet             appartement                 0   
                                                   maison                      0   

                                                                 années jugées  \
modèle                                             type de bien                  
H3a – Régression linéaire – prix N-1 seul          appartement               2   
                                                   maison                    2   
H3a – Régression linéaire – prix N-1 + densité     appartement               2   
                                                   maison                    2   
H3a – Régression linéaire – prix N-1 + densité ... appartement               2   
                                                   maison                    2   
H3a – Régression linéaire – prix N-1 + densité ... appartement               2   
                                                   maison                    2   
H3a – Régression linéaire – bloc complet           appartement               2   
                                                   maison                    2   
H3a – Lasso – bloc complet                         appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – prix N-1 seul            appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – prix N-1 + densité       appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – prix N-1 + densité + ... appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – prix N-1 + densité + ... appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – bloc complet             appartement               2   
                                                   maison                    2   

                                                                 retenu  
modèl

**Comment lire le tableau :** pour chaque modèle et type de bien, le nombre d'années jugées réussies (sur 2). Une année est réussie si l'intervalle du bootstrap est entièrement sous 0, la rang-bisériale négative et le gain sur la naïve d'au moins 3 %. Un modèle est retenu s'il réussit la majorité des années jugées (ici, les deux).

**Résultats :**
- **Aucun modèle n'est retenu** : aucune année réussie, pour les 11 modèles et les deux types de bien.
- **Les modèles font nettement moins bien que la naïve**, surtout pour les maisons en 2024 : de 194 à 217 €/m² d'erreur contre 129 €/m² pour la naïve (gain de −50 à −68 %), et le modèle ne fait mieux que dans 18 à 22 % des sous-bassins.
- **Seule exception : les appartements en 2023**, où l'intervalle contient 0 pour les deux références (régression et boosting sur le prix N-1 seul), la densité, les équipements et le Lasso : on ne peut pas dire qu'ils fassent moins bien que la naïve.
- **Même cause que dans H3b** : le modèle « prix N-1 seul » perd déjà contre la naïve (de −10 à −68 %). Le problème ne vient pas du bloc, mais du retournement du marché (section 6.1).

### 5.2 Le bloc apporte-t-il quelque chose au prix N-1 seul ?
Contre la naïve, le retournement du marché fait perdre tous les modèles (voir H3b et la section 6.1). La question propre à l'hypothèse est autre : l'attractivité améliore-t-elle la prévision **par rapport au même modèle sur le prix N-1 seul** ?

On applique les mêmes mesures et la même règle de décision, en remplaçant la naïve par la référence :
- **régressions et Lasso** : la régression linéaire sur le prix N-1 seul (4.1) ;
- **gradient boosting** : le boosting sur le prix N-1 seul (4.7).

`evaluer` et `tester` lisent la prévision de comparaison dans la colonne `prix_n1` : on leur donne une copie de l'année jugée où cette colonne contient la prévision de la référence.

In [18]:
def reference_de(nom):
    """Référence d'un modèle : le boosting sur prix N-1 seul pour un boosting, la régression sinon."""
    if "Gradient boosting" in nom:
        return REFERENCE_GB
    return REFERENCE


resultats_ref, tests_ref = [], []
modeles_compares = [n for n in dict.fromkeys(k[0] for k in previsions) if n not in [REFERENCE, REFERENCE_GB]]

for nom in modeles_compares:
    for annee, (app, ev) in FENETRES.items():
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = previsions[(reference_de(nom), annee)]     # la comparaison se fait avec la référence
        evaluer(nom, ev_ref, previsions[(nom, annee)], resultats_ref)
        tester(nom, ev_ref, previsions[(nom, annee)], tests_ref)

# Mêmes tableaux que contre la naïve, avec « référence » à la place de « naïve »
res_ref = pd.DataFrame(resultats_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_ref = pd.DataFrame(tests_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_ref[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                 "gain sur la référence (%)", "bassins où le modèle gagne (%)"]].round(2))
display(tst_ref[["modèle", "type de bien", "année jugée", "écart de MAE avec la référence (€/m²)",
                 "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3))

,modèle,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),bassins où le modèle gagne (%)
0,H3a – Régression linéaire – prix N-1 + densité,appartement,2023,186.83,185.46,-0.74,48.43
1,H3a – Régression linéaire – prix N-1 + densité,maison,2023,169.99,170.00,0.01,58.54
2,H3a – Régression linéaire – prix N-1 + densité,appartement,2024,220.86,221.51,0.29,46.17
3,H3a – Régression linéaire – prix N-1 + densité,maison,2024,214.50,214.92,0.20,60.59
4,H3a – Régression linéaire – prix N-1 + densité...,appartement,2023,186.44,185.46,-0.53,49.71
5,H3a – Régression linéaire – prix N-1 + densité...,maison,2023,167.62,170.00,1.40,58.09
6,H3a – Régression linéaire – prix N-1 + densité...,appartement,2024,222.22,221.51,-0.32,51.03
7,H3a – Régression linéaire – prix N-1 + densité...,maison,2024,209.78,214.92,2.39,59.97
8,H3a – Régression linéaire – prix N-1 + densité...,appartement,2023,193.48,185.46,-4.32,42.04
9,H3a – Régression linéaire – prix N-1 + densité...,maison,2023,186.11,170.00,-9.48,30.67


,modèle,type de bien,année jugée,écart de MAE avec la référence (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale
0,H3a – Régression linéaire – prix N-1 + densité,appartement,2023,1.366,0.495,2.561,0.000,0.168
1,H3a – Régression linéaire – prix N-1 + densité,maison,2023,-0.014,-0.376,0.380,0.000,-0.151
2,H3a – Régression linéaire – prix N-1 + densité,appartement,2024,-0.647,-2.786,0.998,0.294,0.042
3,H3a – Régression linéaire – prix N-1 + densité,maison,2024,-0.423,-0.933,0.190,0.000,-0.188
4,H3a – Régression linéaire – prix N-1 + densité...,appartement,2023,0.977,-3.377,6.674,0.835,-0.008
5,H3a – Régression linéaire – prix N-1 + densité...,maison,2023,-2.376,-3.418,-1.251,0.000,-0.182
6,H3a – Régression linéaire – prix N-1 + densité...,appartement,2024,0.709,-3.740,6.299,0.284,-0.043
7,H3a – Régression linéaire – prix N-1 + densité...,maison,2024,-5.134,-6.941,-3.094,0.000,-0.274
8,H3a – Régression linéaire – prix N-1 + densité...,appartement,2023,8.018,3.318,12.331,0.000,0.202
9,H3a – Régression linéaire – prix N-1 + densité...,maison,2023,16.109,13.234,18.744,0.000,0.495


#### Règle de décision contre la référence
Même règle que contre la naïve, avec le modèle « prix N-1 seul » à la place de la naïve.

In [19]:
regle_decision(res_ref, tst_ref, comparaison="référence")

années réussies  \
modèle                                             type de bien                    
H3a – Régression linéaire – prix N-1 + densité     appartement                 0   
                                                   maison                      0   
H3a – Régression linéaire – prix N-1 + densité ... appartement                 0   
                                                   maison                      0   
H3a – Régression linéaire – prix N-1 + densité ... appartement                 1   
                                                   maison                      1   
H3a – Régression linéaire – bloc complet           appartement                 1   
                                                   maison                      1   
H3a – Lasso – bloc complet                         appartement                 2   
                                                   maison                      1   
H3a – Gradient boosting – prix N-1 + densité       appartement                 0   
                                                   maison                      0   
H3a – Gradient boosting – prix N-1 + densité + ... appartement                 0   
                                                   maison                      0   
H3a – Gradient boosting – prix N-1 + densité + ... appartement                 1   
                                                   maison                      1   
H3a – Gradient boosting – bloc complet             appartement                 1   
                                                   maison                      1   

                                                                 années jugées  \
modèle                                             type de bien                  
H3a – Régression linéaire – prix N-1 + densité     appartement               2   
                                                   maison                    2   
H3a – Régression linéaire – prix N-1 + densité ... appartement               2   
                                                   maison                    2   
H3a – Régression linéaire – prix N-1 + densité ... appartement               2   
                                                   maison                    2   
H3a – Régression linéaire – bloc complet           appartement               2   
                                                   maison                    2   
H3a – Lasso – bloc complet                         appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – prix N-1 + densité       appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – prix N-1 + densité + ... appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – prix N-1 + densité + ... appartement               2   
                                                   maison                    2   
H3a – Gradient boosting – bloc complet             appartement               2   
                                                   maison                    2   

                                                                 retenu  
modèle                                             type de bien          
H3a – Régression linéaire – prix N-1 + densité     appartement    False  
                                                   maison         False  
H3a – Régression linéaire – prix N-1 + densité ... appartement    False  
                                                   maison         False  
H3a – Régression linéaire – prix N-1 + densité ... appartement    False  
                                                   maison         False  
H3a – Régression linéaire – bloc complet           appartement    False  
                                                   maison         False  
H3a

**Comment lire le tableau :** même règle qu'en 5.1, avec la référence à la place de la naïve (la régression sur le prix N-1 seul pour les régressions et le Lasso, le boosting sur le prix N-1 seul pour les boostings). Un modèle retenu ici fait vraiment mieux que le même modèle sans le bloc, sur les deux années jugées.

**Résultats contre le modèle « prix N-1 seul » :**
- **La densité seule n'apporte rien** : écart de moins de 1 % avec la référence, en régression comme en boosting, pour les deux types de bien et les deux années.
- **Les équipements apportent un gain faible, pour les maisons seulement** : de 1,4 % (2023) à 2,4 % (2024) en régression, 2,9 % en 2024 en boosting. L'intervalle est entièrement sous 0, mais le gain reste sous le seuil de 3 % : non retenu. Pour les appartements, rien.
- **La délinquance change de sens d'une année à l'autre.** Elle fait perdre en 2023 (de −4 à −11 % : par exemple +16 €/m² d'erreur pour les maisons en régression, IC [+13 ; +19]) et gagner en 2024 (de +4 à +11 %, IC entièrement sous 0). Une seule année réussie sur deux, pour la régression comme pour le boosting et pour les deux types de bien : non retenue. Le bloc complet suit exactement le même schéma, porté par la délinquance.
- **Un seul modèle est formellement retenu : le Lasso, pour les appartements** (gain de 6,2 % en 2023 et 9,6 % en 2024). **Mais ce gain ne vient pas du bloc** (section 6.2) : le Lasso met à zéro presque toutes les variables d'attractivité. Pour les maisons en 2024, il les écarte toutes et gagne quand même 7,7 % sur la référence. Son gain ne vient donc pas de l’attractivité, mais de la régularisation elle-même, appliquée au prix N-1.

## 6. Interprétation

### 6.1 Contexte : le marché s'est-il retourné ?
Les modèles apprennent sur les années de hausse 2020-2022, puis sont jugés sur 2023 et 2024. Pour lire leurs résultats, on mesure l'évolution du prix d'une année sur l'autre (prix / prix N-1 − 1), en moyenne et en médiane, par année et par type de bien, jusqu'à 2024. L'année 2025, réservée au test final, n'est pas regardée.

In [20]:
jusqua_2024 = table[table["annee"] <= commun.ANNEE_VALIDATION]
evolution = (jusqua_2024[CIBLE] / jusqua_2024["prix_n1"] - 1) * 100
retournement = (evolution.groupby([jusqua_2024["annee"], jusqua_2024.index.get_level_values("type_bien")])
                .agg(["mean", "median"]).round(1)
                .rename(columns={"mean": "évolution moyenne (%)", "median": "évolution médiane (%)"}))
retournement.unstack()

évolution moyenne (%)        évolution médiane (%)       
type_bien           appartement maison           appartement maison
annee                                                              
2017                        2.2    1.8                   2.1    2.0
2018                        1.4    2.1                   1.5    2.1
2019                        3.2    2.3                   3.0    2.3
2020                        5.2    4.7                   5.1    4.7
2021                        8.0    7.9                   7.5    7.7
2022                        8.8    9.6                   8.2    9.1
2023                        2.9    1.3                   2.9    0.8
2024                       -0.8   -2.4                  -1.5   -2.9

**Constat :**
- **2017-2019 : hausse lente** (+1,4 à +3,2 % par an) ; **2020-2022 : forte hausse** (+4,7 à +9,6 % par an), ce sont les années d'apprentissage.
- **2023 : net ralentissement** (+2,9 % pour les appartements, +1,3 % pour les maisons) ; **2024 : baisse** (−0,8 % et −2,4 % en moyenne). Ce sont les années jugées.
- Un modèle qui prolonge la hausse apprise surestime le prix ; la naïve (prix inchangé) n'est pas touchée. Les variables du bloc sont des photos fixes (équipements, délinquance lissée sur 3 ans) : elles ne peuvent pas annoncer un retournement.

### 6.2 Quelles variables le Lasso garde-t-il ?
Coefficients du Lasso, par type de bien et année jugée. Les variables sont en log puis mises à l'échelle : les coefficients se comparent entre eux. Un coefficient de 0 veut dire que la variable est écartée. Signe positif : la variable fait monter le prix prévu, à prix N-1 égal.

In [21]:
coefficients = []
for (nom, annee, type_bien), (m, X_ev, y_ev) in ajustes.items():
    if nom == LASSO:
        # GridSearchCV -> meilleur modèle (log de la cible) -> pipeline -> Lasso
        lasso = m.best_estimator_.regressor_.named_steps["modele"]
        coefficients.append(pd.Series(lasso.coef_, index=X_ev.columns, name=type_bien + " " + str(annee)))
coefficients = pd.concat(coefficients, axis=1)
print("alpha retenu :", {k[1:]: v[0].best_params_ for k, v in ajustes.items() if k[0] == LASSO})
coefficients.round(4)

alpha retenu : {(2023, 'appartement'): {'regressor__modele__alpha': 0.01}, (2023, 'maison'): {'regressor__modele__alpha': 0.0001}, (2024, 'appartement'): {'regressor__modele__alpha': 0.01}, (2024, 'maison'): {'regressor__modele__alpha': 0.01}}


,appartement 2023,maison 2023,appartement 2024,maison 2024
prix_n1,0.5198,0.4623,0.5194,0.4594
densite_population,0.0000,0.0141,0.0000,0.0000
taux_equip_commerces,0.0000,-0.0025,0.0000,0.0000
taux_equip_sante,0.0000,0.0018,0.0000,0.0000
taux_equip_services,0.0000,0.0059,0.0000,0.0000
taux_equip_sports,-0.0000,-0.0049,-0.0000,-0.0000
taux_equip_tourisme,0.0002,0.0056,0.0042,0.0000
taux_equip_transports,0.0000,-0.0014,0.0000,0.0000
taux3_vols_vehicules,-0.0000,0.0003,0.0000,0.0000
taux3_vols_violents,0.0000,-0.0087,0.0000,0.0000


**Constat :**
- **Le Lasso garde presque uniquement le prix N-1.** Pour les appartements, une seule variable du bloc survit, les équipements de tourisme, avec un coefficient minime (0,0002 en 2023, 0,004 en 2024, contre 0,52 pour le prix N-1). Pour les maisons en 2024, toutes les variables du bloc sont à zéro.
- **Seule exception : les maisons en 2023**, où le réglage retenu est le plus faible de la grille (alpha = 0,0001) : le Lasso garde alors toutes les variables, avec de petits coefficients, et se comporte comme la régression du bloc complet (même erreur, 184 €/m²).
- **Conclusion :** le gain du Lasso en section 5.2 ne vient pas de l’attractivité : les variables du bloc sont à zéro, sauf pour les maisons en 2023. Il vient de la régularisation, qui modifie le coefficient du prix N-1 par rapport à la régression simple. Pour le confirmer, il faudrait comparer à un Lasso sur le prix N-1 seul.

### 6.3 Importance des variables
Importance par permutation, sur l'année jugée : on mélange au hasard les valeurs d'une variable entre les sous-bassins, 10 fois, et on mesure de combien l'erreur du modèle change. Modèle étudié : le gradient boosting sur le bloc complet, pour les appartements en 2024 (le type de bien où un gain est attendu).

In [22]:
choix = (COMPLET_GB, 2024, "appartement")        # modèle, année jugée, type de bien étudiés
m, X_ev, y_ev = ajustes[choix]
imp = permutation_importance(m, X_ev, y_ev, scoring="neg_mean_absolute_error",
                             n_repeats=10, random_state=commun.GRAINE)
pd.DataFrame({"hausse de la MAE (€/m²)": imp.importances_mean,
              "écart-type (€/m²)": imp.importances_std},
             index=X_ev.columns).sort_values("hausse de la MAE (€/m²)", ascending=False).round(1)

,hausse de la MAE (€/m²),écart-type (€/m²)
prix_n1,1524.8,29.2
taux_equip_transports,3.1,0.7
taux3_vols_violents,1.7,0.3
taux3_stupefiants,1.1,0.5
taux3_vols_vehicules,-0.5,0.4
densite_population,-0.7,0.8
taux_equip_sports,-0.9,0.6
taux3_violences_sexuelles,-1.0,0.3
taux3_cambriolages,-1.1,1.1
taux_equip_sante,-2.0,0.3


**Comment lire le tableau :** une valeur positive veut dire que mélanger la variable fait monter l'erreur : le modèle s'en sert utilement. Négative : la variable dessert le modèle sur cette année. Une variable compte si sa valeur dépasse nettement deux écarts-types.

**Constat (gradient boosting, bloc complet, appartements, 2024) :**
- **Le prix N-1 porte presque tout** : le brouiller fait monter l'erreur de 1 525 €/m².
- **Trois variables du bloc aident un peu** : les équipements de transport (+3,1 €/m²), les vols violents (+1,7) et les stupéfiants (+1,1), toutes au-delà de deux écarts-types.
- **Plusieurs desservent le modèle en 2024** : équipements de tourisme (−8,0 €/m²), vols sans violence (−6,3), violences intrafamiliales (−4,0), commerces (−2,9), services et santé (−2,0). Sur cette année, mélanger ces variables fait baisser l'erreur.
- **À rapprocher de la section 5.2** : le bloc complet bat la référence en 2024 pour les appartements (+4,7 %), alors que la plupart de ses variables, prises une à une, desservent le modèle. Les deux mesures ne répondent pas à la même question : la 5.2 compare deux modèles entraînés différemment, la permutation brouille une variable dans un modèle déjà entraîné, les autres variables, liées entre elles, restant en place.

### 6.4 Analyse des erreurs selon le contrôle qualité
L'erreur est-elle plus grande là où les données sont estimées ou complétées ? Pour chaque variable de contrôle qualité : groupe « part nulle », puis quartiles parmi les parts non nulles. Par groupe : erreur moyenne en €/m² et en % du prix (l'erreur en €/m² grandit avec le niveau des prix), et erreur moyenne signée (positive : le modèle surestime). Rho de Spearman entre la variable et l'erreur, en €/m² et en %. Même modèle qu'en 6.3.

In [23]:
CONTROLE = ["part_indicateurs_estimes", "part_logements_taux_impute", "part_menages_couverts"]

m, X_ev, y_ev = ajustes[choix]
erreur = pd.Series(m.predict(X_ev), index=X_ev.index) - y_ev       # signée : positive = prix surestimé
erreur_pct = erreur.abs() / y_ev * 100                             # erreur absolue en % du prix réel
controle = table.loc[X_ev.index, CONTROLE]

for v in CONTROLE:
    x = controle[v]
    groupe = pd.Series("part nulle", index=x.index)
    non_nul = x > 0
    if non_nul.sum() >= 4:
        groupe[non_nul] = pd.qcut(x[non_nul], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop").astype(str)
    tableau = pd.DataFrame({"MAE (€/m²)": erreur.abs(), "erreur en % du prix": erreur_pct,
                            "erreur signée (€/m²)": erreur}).groupby(groupe).mean()
    tableau["lignes"] = groupe.value_counts()
    rho, p = stats.spearmanr(x, erreur.abs())
    rho_pct, _ = stats.spearmanr(x, erreur_pct)
    print(f"\n{v} : rho avec l'erreur en €/m² = {rho:.2f} (p = {p:.3f}) | rho avec l'erreur en % = {rho_pct:.2f}")
    print(tableau.round(1).to_string())


part_indicateurs_estimes : rho avec l'erreur en €/m² = -0.11 (p = 0.001) | rho avec l'erreur en % = 0.02
            MAE (€/m²)  erreur en % du prix  erreur signée (€/m²)  lignes
Q1               224.5                  7.5                 200.8     194
Q2               164.7                  7.3                 128.1     194
Q3               192.7                  8.8                 138.3     194
Q4               226.6                  8.8                 152.1     194
part nulle       380.3                  7.6                 378.6      47

part_logements_taux_impute : rho avec l'erreur en €/m² = -0.30 (p = 0.000) | rho avec l'erreur en % = -0.01
            MAE (€/m²)  erreur en % du prix  erreur signée (€/m²)  lignes
Q1               229.8                  7.8                 181.0     135
Q2               196.0                  8.3                 136.0     134
Q3               146.5                  8.8                 100.1     134
Q4               148.9                  8.3  

**Comment lire :** pour chaque variable de contrôle qualité, la MAE, l'erreur en % du prix et l'erreur signée par groupe (part nulle, puis quartiles). Un rho proche de 0 veut dire que l'erreur ne dépend pas de la qualité des données. `part_indicateurs_estimes` est la plus utile ici : elle dit si les indicateurs de délinquance ont été estimés.

**Constat (gradient boosting, bloc complet, appartements, 2024) :**
- **En €/m², l'erreur est plus grande là où les données sont les plus complètes** : 380 €/m² pour les 47 sous-bassins sans aucun indicateur de délinquance estimé, 272 €/m² quand aucun taux de vacance n'est complété. Ce sont les territoires les plus grands et les plus urbains, où les prix sont les plus élevés.
- **En % du prix, l'erreur est presque la même partout** : de 7,3 à 8,8 % selon les groupes, pour les trois variables de contrôle (rho avec l'erreur en % de −0,01 à 0,02). L'écart en €/m² vient du niveau des prix, pas de la qualité des données.
- **Le modèle surestime le prix dans tous les groupes** (erreur signée de +100 à +379 €/m²), ce qui est cohérent avec le retournement du marché en 2024.

**Conclusion :** l'erreur n'est pas concentrée dans les territoires où la délinquance est estimée. La qualité des données n'est pas une limite pour ce modèle.

## 7. Conclusion

**Question :** ajoutée au prix N-1, l'attractivité (densité, équipements, délinquance) améliore-t-elle la prévision du prix médian au m² ? Attendu : un petit gain pour les appartements, aucun pour les maisons.

**1. Contre la naïve : aucun modèle n'est retenu.** Les 11 modèles perdent, surtout pour les maisons en 2024 (gain de −50 à −68 %). Seule exception, les appartements en 2023, où plusieurs modèles sont à égalité avec la naïve. Cause : le retournement du marché, comme dans H3b ; le modèle « prix N-1 seul » perd déjà.

**2. Contre le modèle « prix N-1 seul » : le bloc attractivité n'est pas retenu.**
- **Densité seule** : aucun apport (moins de 1 %).
- **Équipements** : un gain faible pour les maisons (1,4 à 2,9 %), réel mais sous le seuil de 3 % ; rien pour les appartements.
- **Délinquance** : elle change de sens d'une année à l'autre (de −4 à −11 % en 2023, de +4 à +11 % en 2024), en régression comme en boosting. Son effet n'est pas stable : non retenue. Les territoires où elle est élevée ont un peu moins monté pendant les années de hausse (section 2.2) ; le modèle prolonge cet écart, ce qui pénalise 2023 et aide 2024, quand le marché recule. Ce mécanisme reste une hypothèse, non vérifiée.
- **Lasso** : Lasso : formellement retenu pour les appartements (gain de 6 à 10 %), mais il écarte presque toutes les variables du bloc. Son gain vient de la régularisation du prix N-1, pas de l’attractivité (section 6.2).

**3. L'attendu de la feuille de route est confirmé pour les maisons (aucun gain utile), pas pour les appartements** : le petit gain attendu n'apparaît pas de façon stable.

**4. Qualité des données** : l'erreur n'est pas concentrée là où la délinquance est estimée ; rapportée au prix, elle est de 7 à 9 % dans tous les groupes.

**Bloc transmis à Christian pour le modèle complet :** aucune variable d'attractivité retenue. Deux pistes à tester dans le modèle complet, sans les retenir d'office :
- les équipements pour les maisons (gain proche du seuil) ;
- la régularisation (Lasso / Ridge) sur le prix N-1, dont le gain ne dépend pas du bloc.

**Limites :**
- Plusieurs variantes sont comparées sur les mêmes années jugées : celle qui ressort a pu en profiter en partie. Le test final sur 2025, fait une seule fois sur le modèle retenu, le confirmera.
- L'importance des variables porte sur une seule année et un seul type de bien ; elle décrit une tendance, sans test.
- La densité est aussi testée dans H3b (en variante) : dans le modèle complet, elle n'entre qu'une fois.